# LeetCode 1915 — Number of Wonderful Substrings bằng Python

Link đề: https://leetcode.com/problems/number-of-wonderful-substrings/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Bit Manipulation, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def wonderfulSubstrings(self, word: str) -> int:
        ...
```

Lộ trình: **duyệt mọi chuỗi con + `Counter`** `O(n³)` → **mở rộng chuỗi con với mặt nạ chẵn lẻ chạy** `O(n²)` → **mặt nạ tiền tố + bảng đếm, mỗi vị trí thử 11 mặt nạ** `O(10·n)` (lời giải nộp) → **mảng 1024 phần tử thay cho `dict`** (nhanh hơn hằng số).

## 1. Mục tiêu học

1. Biểu diễn **tính chẵn lẻ số lần xuất hiện** của 10 chữ cái bằng **một số nguyên 10 bit** (bitmask); cập nhật bằng XOR.
2. Áp dụng **"tổng tiền tố + bảng đếm"** cho phép toán XOR: chuỗi con `word[i:j]` có mặt nạ `P[j] ⊕ P[i]`.
3. Đếm số cặp tiền tố có `P[j] ⊕ P[i]` bằng `0` **hoặc** có **đúng một bit 1** — mỗi vị trí chỉ thử `1 + 10` mặt nạ.
4. Nhớ khởi tạo **tiền tố rỗng** (`cnt[0] = 1`) và thứ tự "đếm trước, tăng sau".
5. Kiểm chứng bằng vét cạn ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

Một chuỗi là **wonderful** nếu **tối đa một** chữ cái xuất hiện **lẻ** lần. Ví dụ `"ccjjc"` và `"abab"` là wonderful, còn `"ab"` thì không.

Cho chuỗi `word` chỉ gồm **mười chữ cái đầu** (`'a'` tới `'j'`), trả về **số chuỗi con khác rỗng** là wonderful. Các chuỗi con giống nhau ở vị trí khác nhau được đếm **riêng**.

**Ví dụ 1:** `word = "aba"` → `4` (`"a"`, `"b"`, `"a"`, `"aba"`).

**Ví dụ 2:** `word = "aabb"` → `9` (`"a"`, `"aa"`, `"aab"`, `"aabb"`, `"a"`, `"abb"`, `"b"`, `"bb"`, `"b"`).

**Ví dụ 3:** `word = "he"` → `2`.

**Ràng buộc:** `1 ≤ word.length ≤ 10⁵`, `word` chỉ gồm `'a'`..`'j'`.

In [1]:
import random
import time
from collections import Counter, defaultdict
from typing import List

examples = [("aba", 4), ("aabb", 9), ("he", 2)]
for w, want in examples:
    print(repr(w), "-> kỳ vọng", want)

def is_wonderful(t):
    return sum(v % 2 for v in Counter(t).values()) <= 1

print([is_wonderful(t) for t in ["ccjjc", "abab", "ab", "a", "aabbc"]])

'aba' -> kỳ vọng 4
'aabb' -> kỳ vọng 9
'he' -> kỳ vọng 2
[True, True, False, True, True]


## 3. Kiến thức nền

### 3.1 Mặt nạ chẵn lẻ bằng XOR

Bit `c` của `mask` = tính chẵn lẻ số lần chữ cái `c` đã gặp (1 = lẻ). Gặp chữ `c` thì lật bit: `mask ^= 1 << (ord(c) − ord('a'))`.

In [2]:
mask = 0
for ch in "aabcb":
    mask ^= 1 << (ord(ch) - 97)
    print(f"sau {ch!r}: {mask:010b}  (bit 1 = chữ xuất hiện lẻ lần)")

sau 'a': 0000000001  (bit 1 = chữ xuất hiện lẻ lần)
sau 'a': 0000000000  (bit 1 = chữ xuất hiện lẻ lần)
sau 'b': 0000000010  (bit 1 = chữ xuất hiện lẻ lần)
sau 'c': 0000000110  (bit 1 = chữ xuất hiện lẻ lần)
sau 'b': 0000000100  (bit 1 = chữ xuất hiện lẻ lần)


### 3.2 Mặt nạ có tối đa một bit 1

`mask == 0` (mọi chữ chẵn) hoặc `mask & (mask − 1) == 0` (lũy thừa của 2 — đúng một bit). Gộp: `mask & (mask − 1) == 0` đúng cho cả hai trường hợp.

In [3]:
for m in [0, 1, 2, 3, 4, 6, 512, 513]:
    print(f"{m:010b} -> tối đa 1 bit: {m & (m - 1) == 0}")

0000000000 -> tối đa 1 bit: True
0000000001 -> tối đa 1 bit: True
0000000010 -> tối đa 1 bit: True
0000000011 -> tối đa 1 bit: False
0000000100 -> tối đa 1 bit: True
0000000110 -> tối đa 1 bit: False
1000000000 -> tối đa 1 bit: True
1000000001 -> tối đa 1 bit: False


### 3.3 Mặt nạ tiền tố và XOR của đoạn

`P[0] = 0`, `P[j]` = mặt nạ của `word[:j]`. Vì XOR tự triệt tiêu (`x ⊕ x = 0`), mặt nạ của `word[i:j]` là `P[j] ⊕ P[i]` — giống "tổng đoạn = hiệu hai tổng tiền tố".

In [4]:
w = "aabcb"
P = [0]
for ch in w:
    P.append(P[-1] ^ (1 << (ord(ch) - 97)))
i, j = 1, 4
direct = 0
for ch in w[i:j]:
    direct ^= 1 << (ord(ch) - 97)
print(f"mặt nạ của {w[i:j]!r}: P[{j}]^P[{i}] = {P[j] ^ P[i]:05b}, tính trực tiếp = {direct:05b}")

mặt nạ của 'abc': P[4]^P[1] = 00111, tính trực tiếp = 00111


### 3.4 Đếm cặp bằng bảng tần suất

Muốn đếm số `i < j` với `P[i] = T` (một giá trị cho trước): giữ `cnt[x]` = số tiền tố đã gặp có mặt nạ `x`; khi tới `j`, cộng `cnt[T]` **trước** khi đưa `P[j]` vào bảng.

In [5]:
cnt = Counter({0: 1})
pairs = 0
for x in [3, 1, 3, 0, 3]:
    pairs += cnt[x]
    cnt[x] += 1
print("số cặp (i<j) có P[i] == P[j]:", pairs)

số cặp (i<j) có P[i] == P[j]: 4


### 3.5 `list` cố định 1024 phần tử nhanh hơn `dict`

Mặt nạ chỉ có `2¹⁰ = 1024` giá trị ⇒ dùng `cnt = [0] * 1024` thay cho `Counter` — truy cập bằng chỉ số nhanh hơn băm.

In [6]:
cnt = [0] * 1024
cnt[0] = 1
cnt[5] += 2
print(cnt[:8])

[1, 0, 0, 0, 0, 2, 0, 0]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2 `word = "aabb"`. Tính mặt nạ tiền tố (bit 0 = `a`, bit 1 = `b`):

| j | tiền tố | `P[j]` |
|---|---|---|
| 0 | `""` | `00` |
| 1 | `"a"` | `01` |
| 2 | `"aa"` | `00` |
| 3 | `"aab"` | `10` |
| 4 | `"aabb"` | `00` |

Chuỗi con `word[i:j]` wonderful ⇔ `P[i] ⊕ P[j]` có ≤ 1 bit 1 ⇔ `P[i] = P[j]` **hoặc** `P[i]` và `P[j]` khác nhau đúng **một** bit.

- Cặp `P` bằng nhau: `{0, 2, 4}` đều `00` ⇒ `C(3,2) = 3` cặp; `{1}`, `{3}` đơn lẻ ⇒ 0. Tổng 3 (`"aa"`, `"aabb"`, `"bb"`).
- Cặp khác đúng một bit: `00`–`01`: `3 × 1 = 3` cặp; `00`–`10`: `3 × 1 = 3`; `01`–`10` khác 2 bit ⇒ không. Tổng 6.

`3 + 6 = 9`. ✔

In [7]:
w = "aabb"
P = [0]
for ch in w:
    P.append(P[-1] ^ (1 << (ord(ch) - 97)))
pairs = [(i, j) for j in range(len(P)) for i in range(j) if (P[i] ^ P[j]) & ((P[i] ^ P[j]) - 1) == 0]
print("P =", [f"{p:02b}" for p in P])
print("các cặp (i, j) wonderful:", pairs, "->", [w[i:j] for i, j in pairs])

P = ['00', '01', '00', '10', '00']
các cặp (i, j) wonderful: [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (0, 4), (1, 4), (2, 4), (3, 4)] -> ['a', 'aa', 'a', 'aab', 'b', 'aabb', 'abb', 'bb', 'b']


## 5. Cách 1 — Brute Force / baseline: mọi chuỗi con + `Counter`

In [8]:
def wonderful_brute(word):
    n = len(word)
    return sum(is_wonderful(word[i:j]) for i in range(n) for j in range(i + 1, n + 1))

for w, want in examples:
    print(repr(w), wonderful_brute(w), "| kỳ vọng", want)

'aba' 4 | kỳ vọng 4
'aabb' 9 | kỳ vọng 9
'he' 2 | kỳ vọng 2


**Độ phức tạp:** `n(n+1)/2` chuỗi con × `O(độ dài)` để đếm ⇒ `O(n³)`. `n = 10⁵` là bất khả.

**Nút thắt:** đếm lại từ đầu cho mỗi chuỗi con; khi mở rộng `j → j+1` chỉ có **một** chữ thay đổi tính chẵn lẻ.

In [9]:
w300 = "".join(random.Random(0).choice("abcdefghij") for _ in range(300))
t = time.perf_counter(); wonderful_brute(w300); dt = time.perf_counter() - t
print(f"n=300: {dt:.2f}s -> n=10^5 ước tính ≈ {dt * (10**5 / 300) ** 3 / 3600 / 24 / 365:.0f} năm")

n=300: 0.09s -> n=10^5 ước tính ≈ 0 năm


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** tính chẵn lẻ của 10 chữ cái ⇒ mặt nạ 10 bit; chuỗi con wonderful ⇔ mặt nạ của nó có ≤ 1 bit 1.

**Insight 2:** mặt nạ chuỗi con `word[i:j]` = `P[i] ⊕ P[j]`. Với mỗi `j`, số `i < j` hợp lệ =
- số `i` có `P[i] = P[j]` (mặt nạ đoạn = 0), **cộng**
- với mỗi bit `b ∈ [0, 10)`: số `i` có `P[i] = P[j] ⊕ (1 << b)` (mặt nạ đoạn = đúng bit `b`).

**Quy tắc:** duyệt `j` từ trái sang phải, giữ `cnt[x]` = số tiền tố `P[0..j−1]` bằng `x`, với `cnt[0] = 1` ban đầu (tiền tố rỗng). Cộng `cnt[P] + Σ_b cnt[P ⊕ (1 << b)]`, **rồi** `cnt[P] += 1`.

**Invariant:** trước khi xử lý vị trí `j`, `cnt` chứa đúng tần suất các mặt nạ `P[0], …, P[j−1]`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mở rộng chuỗi con với mặt nạ chạy (`O(n²)`)

In [10]:
def wonderful_quadratic(word):
    n, ans = len(word), 0
    for i in range(n):
        mask = 0
        for j in range(i, n):
            mask ^= 1 << (ord(word[j]) - 97)
            if mask & (mask - 1) == 0:
                ans += 1
    return ans

for w, want in examples:
    print(wonderful_quadratic(w), want)

4 4
9 9
2 2


**Độ phức tạp:** `O(n²)` — `5·10⁹` phép với `n = 10⁵`, vẫn quá chậm. Nút thắt: xét **mọi cặp** `(i, j)`; nhưng thực ra chỉ cần biết **bao nhiêu** `i` có mặt nạ phù hợp ⇒ bảng đếm.

### 7.2 Cách 3 — Mặt nạ tiền tố + `Counter` (`O(10·n)`) — ý tưởng chuẩn

In [11]:
def wonderful_prefix_demo(word):
    cnt = Counter({0: 1})
    mask, ans = 0, 0
    for j, ch in enumerate(word):
        mask ^= 1 << (ord(ch) - 97)
        same = cnt[mask]
        one = sum(cnt[mask ^ (1 << b)] for b in range(10))
        ans += same + one
        print(f"j={j} {ch!r}: mask={mask:04b} | cùng mặt nạ {same} | lệch 1 bit {one} | ans={ans}")
        cnt[mask] += 1
    return ans

print("=>", wonderful_prefix_demo("aabb"))

j=0 'a': mask=0001 | cùng mặt nạ 0 | lệch 1 bit 1 | ans=1
j=1 'a': mask=0000 | cùng mặt nạ 1 | lệch 1 bit 1 | ans=3
j=2 'b': mask=0010 | cùng mặt nạ 0 | lệch 1 bit 2 | ans=5
j=3 'b': mask=0000 | cùng mặt nạ 2 | lệch 1 bit 2 | ans=9
=> 9


In [12]:
class SolutionCounter:
    def wonderfulSubstrings(self, word: str) -> int:
        cnt = Counter({0: 1})
        mask = ans = 0
        for ch in word:
            mask ^= 1 << (ord(ch) - 97)
            ans += cnt[mask]
            for b in range(10):
                ans += cnt[mask ^ (1 << b)]
            cnt[mask] += 1
        return ans

for w, want in examples:
    print(SolutionCounter().wonderfulSubstrings(w), want)

4 4
9 9
2 2


**Độ phức tạp:** mỗi vị trí `11` lần tra bảng ⇒ `O(11·n)` thời gian, `O(min(n, 1024))` bộ nhớ.

### 7.3 Cách 4 — Mảng 1024 phần tử (cùng thuật toán, nhanh hơn hằng số)

In [13]:
class SolutionArray:
    def wonderfulSubstrings(self, word: str) -> int:
        cnt = [0] * 1024
        cnt[0] = 1
        bits = [1 << b for b in range(10)]
        mask = ans = 0
        for ch in word:
            mask ^= 1 << (ord(ch) - 97)
            ans += cnt[mask]
            for bit in bits:
                ans += cnt[mask ^ bit]
            cnt[mask] += 1
        return ans

for w, want in examples:
    print(SolutionArray().wonderfulSubstrings(w), want)

4 4
9 9
2 2


### 7.4 Cách 5 — Đếm "offline": tần suất mặt nạ trước, ghép cặp sau

Vì điều kiện **đối xứng** (`P[i] ⊕ P[j]`), có thể đếm tần suất tất cả mặt nạ tiền tố trước rồi ghép: cặp cùng mặt nạ `C(f, 2)`; cặp lệch một bit `f[x] · f[x ⊕ bit]` (mỗi cặp không thứ tự đếm hai lần ⇒ chia 2).

In [14]:
class SolutionOffline:
    def wonderfulSubstrings(self, word: str) -> int:
        f = [0] * 1024
        mask = 0
        f[0] = 1
        for ch in word:
            mask ^= 1 << (ord(ch) - 97)
            f[mask] += 1
        same = sum(v * (v - 1) // 2 for v in f)
        diff = sum(f[x] * f[x ^ (1 << b)] for x in range(1024) if f[x] for b in range(10)) // 2
        return same + diff

for w, want in examples:
    print(SolutionOffline().wonderfulSubstrings(w), want)

4 4
9 9
2 2


**Độ phức tạp:** `O(n + 1024·10)`. Cách này cho thấy rõ bản chất "đếm cặp tiền tố"; bản online (Cách 3/4) tiện hơn khi cần mở rộng (ví dụ giới hạn độ dài).

## 8. TRACE từng bước

Trace Cách 4 trên Ví dụ 1 `"aba"` (chỉ in 2 bit thấp):

| j | ký tự | `mask` | `cnt[mask]` | `Σ cnt[mask ⊕ bit]` | cộng | `ans` | `cnt` sau |
|---|---|---|---|---|---|---|---|
| — | | `00` | | | | 0 | `{00:1}` |
| 0 | a | `01` | 0 | `cnt[00] = 1` | 1 | 1 | `{00:1, 01:1}` |
| 1 | b | `11` | 0 | `cnt[10] + cnt[01] = 0 + 1` | 1 | 2 | `+{11:1}` |
| 2 | a | `10` | 0 | `cnt[11] + cnt[00] = 1 + 1` | 2 | **4** | `+{10:1}` |

Ở `j = 2`: hai chuỗi con mới là `"a"` (khớp tiền tố `"ab"`, mặt nạ `11` lệch 1 bit) và `"aba"` (khớp tiền tố rỗng `00`). ✔

In [15]:
def trace_array(word, nbits=10):
    cnt = Counter({0: 1}); mask = ans = 0
    P = [0]
    for j, ch in enumerate(word):
        mask ^= 1 << (ord(ch) - 97); P.append(mask)
        hits = [i for i in range(j + 1) if (P[i] ^ mask) & ((P[i] ^ mask) - 1) == 0]
        add = cnt[mask] + sum(cnt[mask ^ (1 << b)] for b in range(nbits))
        ans += add
        print(f"j={j} {ch!r} mask={mask:03b}: cộng {add} (chuỗi con mới: {[word[i:j+1] for i in hits]}) -> ans={ans}")
        cnt[mask] += 1
    return ans

trace_array("aba")

j=0 'a' mask=001: cộng 1 (chuỗi con mới: ['a']) -> ans=1
j=1 'b' mask=011: cộng 1 (chuỗi con mới: ['b']) -> ans=2
j=2 'a' mask=010: cộng 2 (chuỗi con mới: ['aba', 'a']) -> ans=4


4

In [16]:
trace_array("he")

j=0 'h' mask=10000000: cộng 1 (chuỗi con mới: ['h']) -> ans=1
j=1 'e' mask=10010000: cộng 1 (chuỗi con mới: ['e']) -> ans=2


2

## 9. Code nộp LeetCode

In [17]:
class Solution:
    def wonderfulSubstrings(self, word: str) -> int:
        cnt = [0] * 1024
        cnt[0] = 1                         # tiền tố rỗng
        mask = ans = 0
        for ch in word:
            mask ^= 1 << (ord(ch) - 97)    # lật tính chẵn lẻ của chữ ch
            ans += cnt[mask]               # đoạn có mọi chữ chẵn
            for b in range(10):            # đoạn có đúng chữ b lẻ
                ans += cnt[mask ^ (1 << b)]
            cnt[mask] += 1                 # đưa tiền tố hiện tại vào bảng SAU khi đếm
        return ans

for w, want in examples:
    assert Solution().wonderfulSubstrings(w) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1:** mặt nạ (tính chẵn lẻ từng chữ) của `word[i:j]` là `P[i] ⊕ P[j]`. *Chứng minh:* bit `c` của `P[j]` = số lần `c` trong `word[:j]` mod 2; số lần `c` trong `word[i:j]` = hiệu hai số đó, và hiệu mod 2 = XOR của hai bit.

**Bổ đề 2:** `word[i:j]` wonderful ⇔ `P[i] ⊕ P[j] ∈ {0} ∪ {2ᵇ : 0 ≤ b < 10}` (vì chỉ có 10 chữ cái).

**Đếm:** mỗi chuỗi con khác rỗng ứng với **đúng một** cặp `0 ≤ i < j ≤ n`. Ở bước `j` (sau khi đọc `word[j−1]`, mặt nạ hiện tại là `P[j]`), bảng `cnt` chứa đúng `P[0..j−1]` (bất biến). Với mỗi giá trị `T` trong 11 giá trị hợp lệ, số `i < j` có `P[i] = P[j] ⊕ T` là `cnt[P[j] ⊕ T]`. 11 tập đó **rời nhau** (các `T` khác nhau cho `P[i]` khác nhau) ⇒ cộng lại không trùng. Tăng `cnt[P[j]]` **sau** khi đếm bảo đảm không ghép `j` với chính nó (đoạn rỗng).

Tổng qua mọi `j` = số cặp `(i, j)` hợp lệ = số chuỗi con wonderful.

Kiểm chứng Bổ đề 1 bằng máy: với mọi đoạn của nhiều chuỗi ngẫu nhiên, mặt nạ tính trực tiếp từ `Counter` của đoạn bằng `P[i] ⊕ P[j]`.

In [18]:
def mask_of(t):
    return sum(1 << (ord(c) - 97) for c, v in Counter(t).items() if v % 2)

rng_l = random.Random(77)
for _ in range(200):
    w = "".join(rng_l.choice("abcdefghij") for _ in range(rng_l.randint(1, 25)))
    P = [0]
    for ch in w:
        P.append(P[-1] ^ (1 << (ord(ch) - 97)))
    for i in range(len(w)):
        for j in range(i + 1, len(w) + 1):
            assert mask_of(w[i:j]) == P[i] ^ P[j]
print("mặt nạ đoạn = P[i] XOR P[j] trên mọi đoạn của 200 chuỗi ngẫu nhiên")

mặt nạ đoạn = P[i] XOR P[j] trên mọi đoạn của 200 chuỗi ngẫu nhiên


## 11. Phân tích Time / Space

Gọi `n = len(word) ≤ 10⁵`, `σ = 10` chữ cái.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi chuỗi con + Counter | `O(n³)` | `O(σ)` |
| 2. Mặt nạ chạy | `O(n²)` | `O(1)` |
| 3. Tiền tố + Counter | `O(n·(σ + 1))` | `O(2^σ)` |
| 4. Tiền tố + mảng | `O(n·(σ + 1))` | `O(2^σ)` |
| 5. Offline | `O(n + 2^σ · σ)` | `O(2^σ)` |

Đáp án có thể tới `n(n+1)/2 ≈ 5·10⁹` (chuỗi toàn một chữ) — vượt `int32`, Python không lo.

In [19]:
rng = random.Random(1915)
big = "".join(rng.choice("abcdefghij") for _ in range(10**5))
same = "a" * 10**5
for label, w in [("ngẫu nhiên", big), ("toàn 'a'", same)]:
    res = []
    for name, sol in [("Counter", SolutionCounter()), ("Mảng/Nộp", Solution()), ("Offline", SolutionOffline())]:
        t = time.perf_counter(); r = sol.wonderfulSubstrings(w); res.append(r)
        print(f"[{label}] {name}: {r} trong {1000*(time.perf_counter()-t):.0f} ms")
    assert len(set(res)) == 1
t = time.perf_counter(); wonderful_quadratic(big[:2000]); dt = time.perf_counter() - t
print(f"Cách 2 với n=2000: {dt:.2f}s -> n=10^5 ước tính ≈ {dt * 2500:.0f}s")

[ngẫu nhiên] Counter: 53859082 trong 80 ms


[ngẫu nhiên] Mảng/Nộp: 53859082 trong 50 ms


[ngẫu nhiên] Offline: 53859082 trong 7 ms


[toàn 'a'] Counter: 5000050000 trong 104 ms


[toàn 'a'] Mảng/Nộp: 5000050000 trong 51 ms
[toàn 'a'] Offline: 5000050000 trong 5 ms


Cách 2 với n=2000: 0.14s -> n=10^5 ước tính ≈ 346s


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu vét cạn)

In [20]:
sols = [Solution(), SolutionCounter(), SolutionArray(), SolutionOffline()]
edge = examples + [
    ("a", 1),
    ("aa", 3),                       # "a", "a", "aa"
    ("ab", 2),                       # "ab" không wonderful
    ("abc", 3),
    ("abcdefghij", 10),              # mọi chữ khác nhau: chỉ chuỗi 1 ký tự
    ("jjjj", 10),                    # toàn một chữ: mọi chuỗi con
    ("abab", 7),
]
for w, want in edge:
    assert wonderful_brute(w) == want, (w, wonderful_brute(w))
    for s in sols:
        assert s.wonderfulSubstrings(w) == want, (w, s)
    assert wonderful_quadratic(w) == want
    print("OK", repr(w), "->", want)

OK 'aba' -> 4
OK 'aabb' -> 9
OK 'he' -> 2
OK 'a' -> 1
OK 'aa' -> 3
OK 'ab' -> 2
OK 'abc' -> 3
OK 'abcdefghij' -> 10
OK 'jjjj' -> 10
OK 'abab' -> 7


### 12.2 Ngẫu nhiên có seed: so với vét cạn

In [21]:
rng = random.Random(42)
for _ in range(600):
    alpha = rng.choice(["ab", "abc", "abcdefghij", "aj"])
    w = "".join(rng.choice(alpha) for _ in range(rng.randint(1, 40)))
    want = wonderful_brute(w)
    for s in sols:
        assert s.wonderfulSubstrings(w) == want, w
    assert wonderful_quadratic(w) == want
for _ in range(10):
    w = "".join(rng.choice("abcdefghij") for _ in range(rng.randint(500, 1500)))
    assert len({s.wonderfulSubstrings(w) for s in sols} | {wonderful_quadratic(w)}) == 1
print("600 case nhỏ (vét cạn) + 10 case vừa (Cách 2): khớp")

600 case nhỏ (vét cạn) + 10 case vừa (Cách 2): khớp


### 12.3 Kiểm tra cận: chuỗi toàn một chữ cho `n(n+1)/2`

In [22]:
for n in [1, 2, 5, 100, 1000]:
    assert Solution().wonderfulSubstrings("c" * n) == n * (n + 1) // 2
print("toàn một chữ: đúng bằng n(n+1)/2")

toàn một chữ: đúng bằng n(n+1)/2


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên tiền tố rỗng (`cnt[0] = 1`)

In [23]:
def wrong_no_empty(word):
    cnt = [0] * 1024                         # SAI: thiếu cnt[0] = 1
    mask = ans = 0
    for ch in word:
        mask ^= 1 << (ord(ch) - 97)
        ans += cnt[mask] + sum(cnt[mask ^ (1 << b)] for b in range(10))
        cnt[mask] += 1
    return ans

for w, want in examples:
    print(repr(w), "sai:", wrong_no_empty(w), "| đúng:", want)

'aba' sai: 2 | đúng: 4
'aabb' sai: 5 | đúng: 9
'he' sai: 1 | đúng: 2


**Vì sao sai:** thiếu `P[0] = 0` thì mất mọi chuỗi con **bắt đầu ở chỉ số 0** (`"a"` đầu, `"aba"`…). **Sửa:** `cnt[0] = 1`.

### Lỗi 2 — Chỉ đếm trường hợp "mọi chữ chẵn" (quên "đúng một chữ lẻ")

In [24]:
def wrong_only_even(word):
    cnt = Counter({0: 1}); mask = ans = 0
    for ch in word:
        mask ^= 1 << (ord(ch) - 97)
        ans += cnt[mask]                     # SAI: thiếu 10 mặt nạ lệch một bit
        cnt[mask] += 1
    return ans

for w, want in examples:
    print(repr(w), "sai:", wrong_only_even(w), "| đúng:", want)

'aba' sai: 0 | đúng: 4
'aabb' sai: 3 | đúng: 9
'he' sai: 0 | đúng: 2


**Vì sao sai:** "tối đa một chữ lẻ" gồm cả trường hợp **đúng một** chữ lẻ (như `"a"`, `"aba"`). Chỉ đếm mặt nạ bằng nhau là đếm số chuỗi con mà mọi chữ chẵn.

### Lỗi 3 — Tăng `cnt[mask]` **trước** khi đếm

In [25]:
def wrong_order(word):
    cnt = Counter({0: 1}); mask = ans = 0
    for ch in word:
        mask ^= 1 << (ord(ch) - 97)
        cnt[mask] += 1                       # SAI: tự ghép với chính mình (đoạn rỗng)
        ans += cnt[mask] + sum(cnt[mask ^ (1 << b)] for b in range(10))
    return ans

for w, want in examples:
    print(repr(w), "sai:", wrong_order(w), "| đúng:", want)

'aba' sai: 7 | đúng: 4
'aabb' sai: 13 | đúng: 9
'he' sai: 4 | đúng: 2


**Vì sao sai:** `P[j]` ghép với chính nó tạo ra cặp `(j, j)` — chuỗi con **rỗng**, mà đề chỉ đếm chuỗi con khác rỗng. Thừa đúng 1 mỗi vị trí.

### Lỗi 4 — Kiểm "tối đa một chữ lẻ" thành "đúng một chữ lẻ"

In [26]:
def wrong_check(word):
    n, ans = len(word), 0
    for i in range(n):
        mask = 0
        for j in range(i, n):
            mask ^= 1 << (ord(word[j]) - 97)
            if bin(mask).count("1") == 1:    # SAI: bỏ trường hợp mask == 0
                ans += 1
    return ans

for w, want in examples:
    print(repr(w), "sai:", wrong_check(w), "| đúng:", want)

'aba' sai: 4 | đúng: 4
'aabb' sai: 6 | đúng: 9
'he' sai: 2 | đúng: 2


**Vì sao sai:** "tối đa một" gồm cả **không có** chữ nào lẻ (`mask == 0`, như `"aa"`, `"aabb"`). **Sửa:** `bin(mask).count("1") <= 1` hoặc `mask & (mask − 1) == 0`.

### Lỗi 5 — Duyệt mọi cặp tiền tố thay vì tra bảng (đúng nhưng `O(n²)`)

In [27]:
def slow_pairs(word):
    P = [0]
    for ch in word:
        P.append(P[-1] ^ (1 << (ord(ch) - 97)))
    return sum(1 for j in range(len(P)) for i in range(j) if (P[i] ^ P[j]) & ((P[i] ^ P[j]) - 1) == 0)

w = big[:2500]
t = time.perf_counter(); r = slow_pairs(w); dt = time.perf_counter() - t
assert r == Solution().wonderfulSubstrings(w)
print(f"n=2500: {dt:.2f}s (đúng) -> n=10^5 ≈ {dt * 1600:.0f}s; bảng đếm: tức thời")

n=2500: 0.22s (đúng) -> n=10^5 ≈ 348s; bảng đếm: tức thời


**Vì sao chưa tốt:** dùng mặt nạ tiền tố nhưng vẫn ghép **từng cặp**. Điểm mấu chốt là **đếm** qua bảng tần suất.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đếm chuỗi con mà mỗi chữ xuất hiện chẵn/lẻ lần"** + bảng chữ nhỏ ⇒ **bitmask chẵn lẻ + XOR tiền tố**.
- Mẫu "đếm số đoạn có `f(đoạn) = T`" với `f` khả nghịch (tổng, XOR) ⇒ **tiền tố + bảng tần suất**, nhớ phần tử khởi tạo cho tiền tố rỗng (LeetCode 560 *Subarray Sum Equals K* là anh em).
- "Tối đa một bit lệch" ⇒ thử `mask` và `mask ⊕ (1 << b)` cho mọi `b`.
- Không gian mặt nạ nhỏ (`2¹⁰`) ⇒ mảng thay cho `dict`.

## 15. Biến thể của bài

1. **LeetCode 1371 — Longest Substring Containing Vowels in Even Counts:** mặt nạ 5 nguyên âm, lưu **vị trí đầu tiên** của mỗi mặt nạ để tìm đoạn **dài nhất**.
2. **LeetCode 1542 — Find Longest Awesome Substring:** đoạn dài nhất có thể hoán vị thành palindrome (≤ 1 chữ số lẻ) — y hệt bài này nhưng tìm max độ dài (Bài tập 5).
3. **Đếm đoạn mà mọi chữ đều chẵn** (chỉ mặt nạ bằng nhau) — một nửa của bài này.
4. **Đếm đoạn có đúng `k` chữ lẻ** ⇒ thử mọi mặt nạ `T` có `popcount(T) = k` (`C(10, k)` giá trị).
5. **Truy vấn trên đoạn `[L, R]`** (nhiều truy vấn) ⇒ Mo's algorithm với bảng tần suất mặt nạ.

Minh họa biến thể 1 (kiểu LeetCode 1371): **độ dài lớn nhất** của chuỗi con mà mỗi nguyên âm `a, e, i, o, u` xuất hiện **chẵn** lần (chuỗi chữ thường bất kỳ). Cùng kỹ thuật mặt nạ, nhưng lưu **vị trí đầu tiên** thay cho số lần đếm. (Kiểm bằng vét cạn.)

In [28]:
def longest_even_vowels(s):
    bit = {c: 1 << i for i, c in enumerate("aeiou")}
    first = {0: 0}
    mask = best = 0
    for j, ch in enumerate(s, start=1):
        mask ^= bit.get(ch, 0)
        if mask in first:
            best = max(best, j - first[mask])
        else:
            first[mask] = j
    return best

def longest_even_vowels_brute(s):
    best = 0
    for i in range(len(s)):
        for j in range(i + 1, len(s) + 1):
            c = Counter(s[i:j])
            if all(c[v] % 2 == 0 for v in "aeiou"):
                best = max(best, j - i)
    return best

assert longest_even_vowels("eleetminicoworoep") == 13       # ví dụ của LeetCode 1371
assert longest_even_vowels("leetcodeisgreat") == 5
rng_v = random.Random(31)
for _ in range(300):
    s_ = "".join(rng_v.choice("aeioubcx") for _ in range(rng_v.randint(1, 20)))
    assert longest_even_vowels(s_) == longest_even_vowels_brute(s_)
print("biến thể nguyên âm chẵn: OK")

biến thể nguyên âm chẵn: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi chuỗi con | `Counter` từng đoạn | `O(n³)` | `O(σ)` | kiểm chứng |
| 2. Mặt nạ chạy | mở rộng đoạn, XOR | `O(n²)` | `O(1)` | `n` ≤ vài nghìn |
| 3. Tiền tố + Counter | 11 lần tra mỗi vị trí | `O(11n)` | `O(2^σ)` | ý tưởng chuẩn |
| 4. Tiền tố + mảng | như 3, mảng 1024 | `O(11n)` | `O(2^σ)` | **nộp bài** |
| 5. Offline | tần suất rồi ghép cặp | `O(n + 2^σ σ)` | `O(2^σ)` | thấy rõ bản chất đếm cặp |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `"aa"`, `"ab"`, `"aaa"`?

**Bài 2 (tính tay):** Lập bảng `mask`, `cnt` cho `"abab"` và kiểm đáp án 7.

**Bài 3 (điền TODO):** Viết `count_all_even(word)` — số chuỗi con mà **mọi** chữ xuất hiện chẵn lần.

**Bài 4 (tự viết lại):** Viết `count_exactly_k_odd(word, k)` — số chuỗi con có **đúng `k`** chữ xuất hiện lẻ lần (`0 ≤ k ≤ 10`).

**Bài 5 (mini challenge — kiểu LeetCode 1542):** Viết `longest_wonderful(word)` trả về **độ dài lớn nhất** của một chuỗi con wonderful, `O(11·n)`.

In [29]:
def count_all_even(word):
    # TODO
    pass

def count_exactly_k_odd(word, k):
    # TODO
    pass

def longest_wonderful(word):
    # TODO: lưu vị trí xuất hiện đầu tiên của mỗi mặt nạ
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Liệt kê tay | `"aa"`: 3 | `"ab"`: 2, `"aaa"`: 6 |
| 2 | `P = 00, 01, 11, 10, 00` | Ghép cặp cùng / lệch 1 bit | 7 (4 chuỗi một ký tự + `"aba"`, `"bab"`, `"abab"`) |
| 3 | Chỉ tra `cnt[mask]` | Nhớ `cnt[0] = 1` | Đếm trước, tăng sau |
| 4 | Tiền xử lý list `T` có `popcount = k` | Mỗi vị trí cộng `Σ cnt[mask ⊕ T]` | `k = 0` hoặc `1` khớp bài gốc khi cộng lại |
| 5 | `first[mask]` = chỉ số tiền tố **nhỏ nhất** có mặt nạ đó | Độ dài `j − first[x]` với `x ∈ {mask, mask ⊕ bit}` | `first[0] = 0`, chỉ gán khi chưa có |

## 19. Đáp án tham khảo

In [30]:
r1 = [Solution().wonderfulSubstrings(w) for w in ("aa", "ab", "aaa")]
print("Bài 1:", r1); assert r1 == [3, 2, 6]
print("Bài 2:"); assert trace_array("abab") == 7

Bài 1: [3, 2, 6]
Bài 2:
j=0 'a' mask=001: cộng 1 (chuỗi con mới: ['a']) -> ans=1
j=1 'b' mask=011: cộng 1 (chuỗi con mới: ['b']) -> ans=2
j=2 'a' mask=010: cộng 2 (chuỗi con mới: ['aba', 'a']) -> ans=4
j=3 'b' mask=000: cộng 3 (chuỗi con mới: ['abab', 'bab', 'b']) -> ans=7


In [31]:
from itertools import combinations as _comb

def count_all_even(word):
    cnt = [0] * 1024; cnt[0] = 1; mask = ans = 0
    for ch in word:
        mask ^= 1 << (ord(ch) - 97)
        ans += cnt[mask]; cnt[mask] += 1
    return ans

def count_exactly_k_odd(word, k):
    targets = [sum(1 << b for b in bits) for bits in _comb(range(10), k)]
    cnt = [0] * 1024; cnt[0] = 1; mask = ans = 0
    for ch in word:
        mask ^= 1 << (ord(ch) - 97)
        for T in targets:
            ans += cnt[mask ^ T]
        cnt[mask] += 1
    return ans

rng = random.Random(3)
for _ in range(300):
    w = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 25)))
    subs = [w[i:j] for i in range(len(w)) for j in range(i + 1, len(w) + 1)]
    odd_counts = [sum(v % 2 for v in Counter(t).values()) for t in subs]
    assert count_all_even(w) == odd_counts.count(0)
    for k in range(0, 5):
        assert count_exactly_k_odd(w, k) == odd_counts.count(k)
    assert count_exactly_k_odd(w, 0) + count_exactly_k_odd(w, 1) == Solution().wonderfulSubstrings(w)
print("Bài 3, 4: OK", count_all_even("aabb"), count_exactly_k_odd("aabb", 2))

Bài 3, 4: OK 3 1


In [32]:
def longest_wonderful(word):
    first = [-1] * 1024
    first[0] = 0
    mask = best = 0
    for j, ch in enumerate(word, start=1):
        mask ^= 1 << (ord(ch) - 97)
        if first[mask] != -1:
            best = max(best, j - first[mask])
        for b in range(10):
            x = mask ^ (1 << b)
            if first[x] != -1:
                best = max(best, j - first[x])
        if first[mask] == -1:
            first[mask] = j
    return best

def longest_brute(word):
    n = len(word)
    return max(j - i for i in range(n) for j in range(i + 1, n + 1) if is_wonderful(word[i:j]))

rng = random.Random(8)
for _ in range(400):
    w = "".join(rng.choice("abcde") for _ in range(rng.randint(1, 30)))
    assert longest_wonderful(w) == longest_brute(w), w
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK", longest_wonderful("aabbcde"), longest_wonderful("he"))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK 5 1


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
cnt = [0] * 1024; cnt[0] = 1           # tiền tố rỗng
mask = ans = 0
for ch in word:
    mask ^= 1 << (ord(ch) - 97)        # lật bit chẵn lẻ
    ans += cnt[mask]                   # đoạn có mọi chữ chẵn
    for b in range(10):
        ans += cnt[mask ^ (1 << b)]    # đoạn có đúng chữ b lẻ
    cnt[mask] += 1                     # tăng SAU khi đếm
return ans
```

**Checklist:**
- [ ] `cnt[0] = 1` cho tiền tố rỗng?
- [ ] Đếm cả "0 chữ lẻ" **và** "đúng 1 chữ lẻ" (11 mặt nạ)?
- [ ] Đếm trước, tăng `cnt[mask]` sau (không đếm đoạn rỗng)?
- [ ] Bit theo `ord(ch) − ord('a')`, chỉ 10 chữ `a..j`?
- [ ] `O(11·n)`, không ghép từng cặp.